Notebook criado para recuperar os arquivos em csv na pasta raw, transformar em parquet e enviar para a pasta bronze do AWS (e local) 

In [1]:
%load_ext autoreload
%autoreload 2


from IPython import get_ipython
import pandas as pd 
import os
import dotenv
from pathlib import Path
from pandas import DataFrame
from botocore.exceptions import ClientError

# Instala o boto3 biblioteca utilizada para conexão do aws 
try:
    import boto3
except ModuleNotFoundError:
    get_ipython().run_line_magic('pip', 'install boto3')
    import boto3

dotenv.load_dotenv()

# Trata o pacote local 'src' para conseguir extrair as funções uteis
try:
    from src.data.utils import gerar_df_dic, converter_para_parquet_bytes, salvar_parquet_local,salvar_parquet_s3
except ModuleNotFoundError:
    get_ipython().run_line_magic('pip', 'install -e ..')
    from src.data.utils import gerar_df_dic, converter_para_parquet_bytes, salvar_parquet_local,salvar_parquet_s3

In [2]:
# Recuperar as variaveis para acessar o AWS 
AWS_REGION = os.getenv("AWS_REGION")
aws_access_key_id=os.getenv("aws_access_key_id")
aws_secret_access_key=os.getenv("aws_secret_access_key")
aws_session_token=os.getenv("aws_session_token")

# Gerar conexão com o S3 
boto3.setup_default_session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token,  # <--- Não esqueça de pegar este lá no "AWS Details"
    region_name=AWS_REGION
)

configurado o boto 3 iremos utiliza-lo para acessar o nosso AWS atraves das nossas variaveis, iremos verificar se já existe o bucket da fiap-pos-tech-002, caso nao exista iremos cria-lo, se existir só iremos conectar a ele

In [ ]:
s3 = boto3.client('s3')
nome_do_bucket = os.getenv("BUCKET_NAME")

response = s3.list_buckets()
buckets_existentes = [bucket['Name'] for bucket in response['Buckets']]

try:
    # verifica a existencia do bucket da Fiap na AWS conectada
    s3.head_bucket(Bucket=nome_do_bucket)
    print(f"Bucket '{nome_do_bucket}' já existe.")
except ClientError as e:
    # Se der erro 404 (Not Found), significa que o bucket não existe então iremos cria-lo
    error_code = e.response['Error']['Code']
    if error_code == '404':
        s3.create_bucket(Bucket=nome_do_bucket)
        print(f"Bucket '{nome_do_bucket}' criado com sucesso.")
    else:
        # Outro erro
        print(f"Erro ao acessar o bucket: {e}")

Bucket 'fiap-tech-challenge-002' já existe.


Utilizamos a função gerar_df_dic localizada no src\data\utils.py 
a função é utilizada para carregar os csv no Raw e carregar todos os dataframes para que possamos transforma-los em .parquet

Pipeline para salvar localmente em formato parquet na camada bronze: 

## Camada BRONZE

In [4]:
anos = [2023,2024,2025] 
tabelas = ['TS_ALUNO','TS_ITEM','TS_ESTADO','TS_MUNICIPIO']

for ano in anos: 
    for tabela in tabelas:
        print(f'trabalhando com a tabela  {tabela}')
        caminho_df = Path(f"../data/bronze/ano={ano}/dados/{tabela}.parquet")
        caminho_dicionario =  Path(f"../data/bronze/ano={ano}/dicionario/dicionario_{tabela}.parquet")

        df, dic = gerar_df_dic(ano, tabela)
        salvar_parquet_local(df, caminho_df)
        salvar_parquet_local(dic, caminho_dicionario)


trabalhando com a tabela  TS_ALUNO
Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx
Arquivo salvo localmente em: ..\data\bronze\ano=2023\dados\TS_ALUNO.parquet
Arquivo salvo localmente em: ..\data\bronze\ano=2023\dicionario\dicionario_TS_ALUNO.parquet
trabalhando com a tabela  TS_ITEM
Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx
Arquivo salvo localmente em: ..\data\bronze\ano=2023\dados\TS_ITEM.parquet
Arquivo salvo localmente em: ..\data\bronze\ano=2023\dicionario\dicionario_TS_ITEM.parquet
trabalhando com a tabela  TS_ESTADO
Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx
Arquivo salvo localmente em: ..\data\bronze\ano=2023\dados\TS_ESTADO.parquet
Arquivo salvo localmente em: ..\data\bronze\ano=2023\dicionario\dicionario_TS_ESTADO.parquet
trabalhando com a tabela  TS_MUNICIPIO
Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados

c:\users\deth_\carmel capital\tecnologia - geral\lucas\estudos\fiap\postech_ai_scientist\tech_challenge_02\src\data\utils.py:16: DtypeWarning: Columns (13,14,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(path_alunos, sep=";", encoding="latin-1")


Lendo dicionário em: ..\data\raw\dados_2025\DICIONÁRIO\Dicionario_Microdados_AEEB_2025.xlsx
Arquivo salvo localmente em: ..\data\bronze\ano=2025\dados\TS_ALUNO.parquet
Arquivo salvo localmente em: ..\data\bronze\ano=2025\dicionario\dicionario_TS_ALUNO.parquet
trabalhando com a tabela  TS_ITEM
Lendo dicionário em: ..\data\raw\dados_2025\DICIONÁRIO\Dicionario_Microdados_AEEB_2025.xlsx
Arquivo salvo localmente em: ..\data\bronze\ano=2025\dados\TS_ITEM.parquet
Arquivo salvo localmente em: ..\data\bronze\ano=2025\dicionario\dicionario_TS_ITEM.parquet
trabalhando com a tabela  TS_ESTADO
Lendo dicionário em: ..\data\raw\dados_2025\DICIONÁRIO\Dicionario_Microdados_AEEB_2025.xlsx
Arquivo salvo localmente em: ..\data\bronze\ano=2025\dados\TS_ESTADO.parquet
Arquivo salvo localmente em: ..\data\bronze\ano=2025\dicionario\dicionario_TS_ESTADO.parquet
trabalhando com a tabela  TS_MUNICIPIO
Lendo dicionário em: ..\data\raw\dados_2025\DICIONÁRIO\Dicionario_Microdados_AEEB_2025.xlsx
Arquivo salvo local

Pipeline para salvar no S3 na camada bronze

In [5]:
# s3 = boto3.client('s3')
# nome_do_bucket = 'fiap-tech-challenge-002'

# anos = [2023, 2024, 2025] 
# tabelas = ['TS_ALUNO', 'TS_ITEM', 'TS_ESTADO', 'TS_MUNICIPIO']

# for ano in anos: 
#     for tabela in tabelas:
#         print(f'\n--- Trabalhando com a tabela {tabela} ({ano}) ---')
        
#         # 3. Carrega os dataframes originais
#         df, dic = gerar_df_dic(ano, tabela)
        
#         # 4. Converte os DataFrames para bytes em formato Parquet
#         df_bytes = converter_para_parquet_bytes(df)
#         dic_bytes = converter_para_parquet_bytes(dic)
        
#         # 5. Define as chaves (caminhos de pastas) estruturadas no S3
#         chave_df = f"bronze/ano={ano}/dados/{tabela}.parquet"
#         chave_dicionario = f"bronze/ano={ano}/dicionario/dicionario_{tabela}.parquet"
        
#         # 6. Salva no S3
#         print(f'Enviando dados de {tabela} para o S3...')
#         salvar_parquet_s3(s3, nome_do_bucket, chave_df, df_bytes)
        
#         print(f'Enviando dicionário de {tabela} para o S3...')
#         salvar_parquet_s3(s3, nome_do_bucket, chave_dicionario, dic_bytes)